In [1]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 87.6 MB/s eta 0:00:00:00:0100:01


#### Import Packages

In [2]:
import os
import cv2
import rasterio
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from pathlib import Path
from sklearn.cluster import KMeans
from rasterio.windows import Window

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import Adam 
from torch.nn.parameter import Parameter
from torch.utils.data import DataLoader, TensorDataset

In [17]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: Mountpoint must not already contain files

## DINOv2 Feature Extractor

In [9]:
class DINOv2FeatureExtractor:
    """Enhanced DINOv2 feature extractor and segmenter"""

    SUPPORTED_MODELS = ['small', 'base', 'large', 'giant']
    SUPPORTED_METHODS = ['pca', 'feature_norm', 'cosine_similarity', 'multi_scale',
                        'kmeans', 'hierarchical', 'dbscan']

    def __init__(self, model_size: str = 'small', device: str = 'cuda'):
        """
        Initialize DINOv2 segmenter

        Args:
            model_size: Model size ('small', 'base', 'large', 'giant')
            device: Computing device ('cuda' or 'cpu')
        """
        if model_size not in self.SUPPORTED_MODELS:
            raise ValueError(f"Model must be one of {self.SUPPORTED_MODELS}")

        self.device = device if torch.cuda.is_available() else 'cpu'
        if device == 'cuda' and not torch.cuda.is_available():
            print("⚠️  CUDA not available, using CPU")

        self.model_size = model_size
        self.model = None
        self.patch_size = 14
        self.metadata = {}  # Store geospatial metadata

    def load_model(self):
        """Load DINOv2 model with error handling"""
        if self.model is not None:
            return

        model_map = {
            'small' : 'dinov2_vits14',
            'base' : 'dinov2_vitb14',
            'large' : 'dinov2_vitl14',
            'giant' : 'dinov2_vitg14'
        }

        print(f"🔄 Loading DINOv2-{self.model_size}...")
        try:
            model_name = f'{model_map[self.model_size]}'
            self.model = torch.hub.load('facebookresearch/dinov2', model_name)
            self.model = self.model.to(self.device)
            self.model.eval()
            print(f"✓ Model loaded successfully on {self.device}")
        except Exception as e:
            raise RuntimeError(f"Failed to load model: {e}")

    def extract_features(self, image: np.ndarray) -> np.ndarray:
        """
        Extract DINOv2 features from image tile

        Args:
            image: Input image (H, W, 3) - must be divisible by 14

        Returns:
            Feature map (H/14, W/14, feature_dim)
        """
        if self.model is None:
            self.load_model()

        # Ensure divisible by patch size
        h, w = image.shape[:2]
        pad_h = (self.patch_size - (h % self.patch_size)) % self.patch_size
        pad_w = (self.patch_size - (w % self.patch_size)) % self.patch_size

        if pad_h > 0 or pad_w > 0:
            image = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')

        # Convert to tensor
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0)
        image_tensor = image_tensor.float() / 255.0
        image_tensor = image_tensor.to(self.device)

        with torch.no_grad():
            features = self.model.forward_features(image_tensor)
            patch_tokens = features['x_norm_patchtokens']

        # Reshape to spatial format
        h_padded, w_padded = image.shape[:2]
        num_patches_h = h_padded // self.patch_size
        num_patches_w = w_padded // self.patch_size
        feature_dim = patch_tokens.shape[-1]

        feature_map = patch_tokens[0].reshape(num_patches_h, num_patches_w, feature_dim)
        feature_map = feature_map.cpu().numpy()

        # Remove padding
        if pad_h > 0 or pad_w > 0:
            h_remove = pad_h // self.patch_size
            w_remove = pad_w // self.patch_size
            if h_remove > 0:
                feature_map = feature_map[:-h_remove, :, :]
            if w_remove > 0:
                feature_map = feature_map[:, :-w_remove, :]

        return feature_map

## Convolutional Autoencoder

In [3]:
class ConvAutoenoder(nn.Module):
    """
    Convolutional Autoencoder for feature learning 
    Based on Deep Covolutional Embedded Clustering architecture
    """

    def __init__(self, input_dim=1024, latent_dim=10):
        super(ConvAutoenoder, self).__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder 
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 500),
            nn.ReLU(),
            nn.Linear(500, 500),
            nn.ReLU(),
            nn.Linear(500, 2000),
            nn.ReLU(),
            nn.Linear(2000, latent_dim)
        )

        # Decoder 
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 2000),
            nn.ReLU(),
            nn.Linear(2000, 500),
            nn.ReLU(),
            nn.Linear(500, 500),
            nn.ReLU(),
            nn.Linear(500, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return z, x_recon

## Clustering Layer

In [4]:
class ClusteringLayer(nn.Module):
    """
    Clustering layer converts input sample (feature) to soft label, i.e. a vector that represents the probability of the 
    sample belonging to each cluster. The probability is calculated with Student's t-distribution.
    """

    def __init__(self, n_clusters=10, hidden=10, cluster_centers=None, alpha=1.0):
        super(ClusteringLayer, self).__init__()
        self.n_clusters = n_clusters
        self.hidden = hidden
        self.alpha = alpha

        if cluster_centers is None:
            initial_cluster_centers = torch.zeros(
                self.n_clusters, 
                self.hidden, 
                dtype=torch.float
                )
            
            nn.init.xavier_uniform_(initial_cluster_centers)
            
        else:
            self.cluster_centers = Parameter(torch.tensor(cluster_centers, dtype=torch.float))
        
        self.cluster_centers = Parameter(initial_cluster_centers)

    def forward(self, x):
        # Student t-distribution, as same as used in t-SNE algorithm.
        # q_ij = (1 + ||z_i - µ_j||^2 / α)^(-(α+1)/2) / Σ_j'(1 + ||z_i - µ_j'||^2 / α)^(-(α+1)/2)
        # q_ij is the probability of assigning sample i to cluster j

        norm_squared = torch.sum((x.unsqueeze(1) - self.cluster_centers)**2, 2)
        numerator = 1.0 / (1.0 + (norm_squared / self.alpha))
        power = float(self.alpha + 1) / 2
        numerator = numerator ** power
        q = numerator / torch.sum(numerator, dim=1, keepdim=True)  # soft assignments

        return q

## Deep Convolutional Embedded Clustering

In [5]:
class DCEC(nn.Module):
    """
    Deep Convolutional Embedded Clustering
    """
    def __init__(self, input_dim=1024, n_clusters=10, latent_dim=10, alpha=1.0):
        super(DCEC, self).__init__()
        self.input_dim = input_dim
        self.n_clusters = n_clusters
        self.latent_dim = latent_dim
        self.alpha = alpha

        # Autoencoder
        self.autoencoder = ConvAutoenoder(input_dim=input_dim, latent_dim=latent_dim)

        # Clustering layer
        self.clustering_layer = ClusteringLayer(n_clusters=n_clusters, hidden=latent_dim, alpha=alpha)

    def forward(self, x):
        z, x_recon = self.autoencoder(x)
        q = self.clustering_layer(z)
        return z, q, x_recon
    
    def target_distribution(self, q):
        # Compute the target distribution p, as defined in the DEC paper
        # p_ij = (q_ij^2 / f_j) / Σ_j'(q_ij'^2 / f_j')
        # where f_j = Σ_i q_ij

        weight = q ** 2 / q.sum(0)
        return (weight.t() / weight.sum(1)).t()
    
    def pretrain(self, dataloader, epochs=100, lr=0.001, device='cuda'):
        """
        Pretrain the autoencoder
        """
        print("Pretraining autoencoder ....")
        optimizer = Adam(self.autoencoder.parameters(), lr=lr)

        self.autoencoder.train()
        for epoch in range(epochs):
            total_loss = 0
            for batch_idx, (data,) in enumerate(dataloader):
                data = data.to(device)

                optimizer.zero_grad()
                _, x_recon = self.autoencoder(data)
                loss = F.mse_loss(x_recon, data)

                total_loss += loss.item()
                loss.backward()
                optimizer.step()
            
            if (epoch + 1) % 50 == 0:
                avg_loss = total_loss / len(dataloader)
                print(f' Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

        print("✓ Pretraining completed!")
    
    def init_cluster_centers(self, dataloader, device='cuda'):
        """
        Initialize cluster centers using KMeans
        """
        print("Initializing cluster centers with KMeans ....")

        # Get encoded features 
        kmeans = KMeans(n_clusters=self.n_clusters, n_init=20)

        features = []
        self.autoencoder.eval()
        with torch.no_grad():
            for batch_idx, (data,) in enumerate(dataloader):
                data = data.to(device)
                z, _ = self.autoencoder(data)
                features.append(z.cpu().numpy())
        
        features = np.concatenate(features, axis=0)

        # Fit k-means 
        y_pred = kmeans.fit_predict(features)

        # Set cluster centers
        self.clustering_layer.cluster_centers.data = torch.tensor(
            kmeans.cluster_centers_, 
            dtype=torch.float,
            device=device
            )
        print(f" ✓ Cluster centers initialized!")

        return y_pred

def train_dcec(model, dataloader, epochs=100, update_interval=5, 
               tol=0.001, lr=0.001, device='cuda'):
    """
    Train DCEC model 

    Args: 
        model: DCEC model 
        dataloader: DataLoader for training data
        epochs: Number of training epochs
        update_interval: Interval for updating target distribution
        tol: Tolerance for convergence
        lr: Learning rate
        device: Device to run the model on ('cuda' or 'cpu')
    """

    print("Training DCEC model ....")

    optimizer = Adam(model.parameters(), lr=lr)

    model.train()
    
    # Initialize target distribution
    y_pred_last = None

    for epoch in range(epochs):
        # Update target distribution
        if epoch % update_interval == 0:
            # Compute current Q
            model.eval()
            q_list = []
            with torch.no_grad():
                for batch_idx, (data,) in enumerate(dataloader):
                    data = data.to(device)
                    _, q, _ = model(data)
                    q_list.append(q.cpu().numpy())

            q = np.concatenate(q_list, axis=0)

            # compute target distribution P
            p = model.target_distribution(torch.tensor(q, device=device)).cpu().numpy()

            # check stopping criterion
            y_pred = q.argmax(1)
            if y_pred_last is not None:
                delta_label = np.sum(y_pred != y_pred_last).astype(np.float32) / y_pred.shape[0]
                print(f'Epoch {epoch}: delta_label={delta_label:.4f}')
                if delta_label < tol:
                    print(f" Delta label {delta_label:.4f} < tol {tol}, stopping training.")
                    break

            y_pred_last = y_pred

            model.train()
    
        # Training loop
        total_loss = 0
        recon_loss_val = 0
        cluster_loss_val = 0

        for batch_idx, (data,) in enumerate(dataloader):
            data = data.to(device)

            # Get batch indices for target distribution
            batch_start = batch_idx * dataloader.batch_size
            batch_end = min(batch_start + dataloader.batch_size, len(p))
            p_batch = torch.tensor(p[batch_start:batch_end], device=device)

            optimizer.zero_grad()

            # Forward pass
            z, q, x_recon = model(data)

            # Reconstruction loss 
            recon_loss = F.mse_loss(x_recon, data)

            # Clustering loss (KL divergence)
            cluster_loss = F.kl_div(q.log(), p_batch, reduction='batchmean')

            # Total loss 
            loss = recon_loss + cluster_loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            recon_loss_val += recon_loss.item()
            cluster_loss_val += cluster_loss.item()
        
        if (epoch + 1) % 10 == 0:
            avg_loss = total_loss / len(dataloader)
            avg_recon = recon_loss_val / len(dataloader)
            avg_cluster = cluster_loss_val / len(dataloader)
            print(f" Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Recon Loss: {avg_recon:.4f}, Cluster Loss: {avg_cluster:.4f}")
    
    print("✓ Training completed!")

                                        

## Processing Tiles 

In [6]:
def process_tile_with_dcec(tile_image, feature_extractor, n_clusters=20, 
                           pretrain_epochs=200, train_epochs=100, device='cuda'):
    
    print(f" Extracting DINOv2 features for {tile_image.shape[0]}x{tile_image.shape[1]} tile ...")

    # Extract features 
    feature_map = feature_extractor.extract_features(tile_image)
    h_feat, w_feat, feat_dim = feature_map.shape
    print(f"  ✓ Feature map shape: {h_feat}x{w_feat}x{feat_dim}")

    # Flatten features 
    features_flat = feature_map.reshape(-1, feat_dim)

    # Create dataloader
    dataset = TensorDataset(torch.FloatTensor(features_flat))
    dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

    # Create DCEC model 
    print(f" Creating DCEC model with {n_clusters} clusters ....")
    model = DCEC(
        input_dim=feat_dim,
        n_clusters=n_clusters,
        latent_dim=10,
        alpha=1.0
    ).to(device)

    # Pretrain autoencoder
    model.pretrain(
        dataloader,
        epochs=pretrain_epochs,
        device=device
    )

    # Initialize cluster centers
    model.init_cluster_centers(
        dataloader,
        device=device
    )

    # Train DCEC model
    train_dcec(
        model,
        dataloader,
        epochs=train_epochs,
        update_interval=5,
        device=device
    )

    # Get final cluster assignments
    print(" Generating cluster assignments ....")
    model.eval()
    all_labels = []

    with torch.no_grad():
        for batch_idx, (data,) in enumerate(dataloader):
            data = data.to(device)
            _, q, _ = model(data)
            labels = q.argmax(1).cpu().numpy()
            all_labels.append(labels)
    
    labels = np.concatenate(all_labels)

    # Reshape to image 
    cluster_map = labels.reshape(h_feat, w_feat)

    # Upsample to original resolution 
    h_orig, w_orig = tile_image.shape[:2]
    cluster_map_upsampled = cv2.resize(
        cluster_map.astype(np.float32),
        (w_orig, h_orig),
        interpolation=cv2.INTER_NEAREST
    ).astype(np.uint8)

    unique, counts = np.unique(cluster_map_upsampled, return_Counts=True)
    print(f" Cluster distribution")
    for cluster_id, count in zip(unique, counts):
        pct = count / cluster_map_upsampled.size * 100
        print(f" Cluster {cluster_id}: {count:,} pixels ({pct:.1f}%)")
    
    return cluster_map_upsampled

## Save to GEOTIFF

In [7]:
def save_geotiff_with_window(output_path, data, src_dataset, row_off, col_off, description=""):
    """Save tile as GeoTIFF with correct geospatial transform"""
    h, w = data.shape[:2] if len(data.shape) == 3 else data.shape
    
    window = Window(col_off, row_off, w, h)
    transform = rasterio.windows.transform(window, src_dataset.transform)
    
    if len(data.shape) == 2:
        data = data[np.newaxis, :, :]
    elif len(data.shape) == 3:
        data = np.transpose(data, (2, 0, 1))
    
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=h,
        width=w,
        count=data.shape[0],
        dtype=data.dtype,
        crs=src_dataset.crs,
        transform=transform,
        compress='deflate'
    ) as dst:
        dst.write(data)
        if description:
            dst.update_tags(description=description)

## Process Whole Orthomosaic

In [16]:
def process_large_orthomosaic_dcec(
    tiff_path,
    output_dir='dcec_original_results',
    tile_size=4096,
    n_clusters=20,
    model_size='large',
    pretrain_epochs=200,
    train_epochs=100,
    visualize=True
):
    """
    Process large orthomosaic with DINOv2 + original DCEC
    
    Args:
        tiff_path: Path to input orthomosaic
        output_dir: Output directory
        tile_size: Tile size
        n_clusters: Number of clusters
        model_size: DINOv2 model size
        pretrain_epochs: Autoencoder pretraining epochs
        train_epochs: DCEC training epochs
        visualize: Create visualizations
    """
    print("=" * 70)
    print("🚀 DINOv2 + DCEC (Original) Segmentation Pipeline")
    print("=" * 70)
    
    tiff_path = Path(tiff_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # Initialize feature extractor (assuming DINOv2FeatureExtractor class exists)
    feature_extractor = DINOv2FeatureExtractor(model_size=model_size)
    feature_extractor.load_model()
    
    # Open source dataset
    print(f"\n📂 Opening: {tiff_path.name}")
    with rasterio.open(tiff_path) as src:
        print(f"  Size: {src.width} × {src.height}")
        print(f"  Bands: {src.count}")
        print(f"  CRS: {src.crs}")
        
        # Calculate tiles
        n_tiles_w = (src.width + tile_size - 1) // tile_size
        n_tiles_h = (src.height + tile_size - 1) // tile_size
        total_tiles = n_tiles_w * n_tiles_h
        
        print(f"\n📊 Processing {total_tiles} tiles ({n_tiles_h}×{n_tiles_w})")
        print(f"  Tile size: {tile_size}×{tile_size}")
        print(f"  Clusters: {n_clusters}")
        print(f"  Pretrain epochs: {pretrain_epochs}")
        print(f"  DCEC epochs: {train_epochs}")
        
        # Process each tile
        tile_idx = 0
        for tile_row in range(n_tiles_h):
            for tile_col in range(n_tiles_w):
                tile_idx += 1
                
                # Window
                col_off = tile_col * tile_size
                row_off = tile_row * tile_size
                width = min(tile_size, src.width - col_off)
                height = min(tile_size, src.height - row_off)
                
                window = Window(col_off, row_off, width, height)
                
                print(f"\n{'='*70}")
                print(f"📍 Tile {tile_idx}/{total_tiles} - Row {tile_row}, Col {tile_col}")
                print(f"  Position: ({col_off}, {row_off})")
                print(f"  Size: {width}×{height}")
                print(f"{'='*70}")
                
                # Read tile
                tile_data = src.read(window=window)
                tile_data = np.transpose(tile_data, (1, 2, 0))
                
                # Handle different data types
                if tile_data.dtype == np.uint16:
                    tile_data = (tile_data / 256).astype(np.uint8)
                elif tile_data.dtype in [np.float32, np.float64]:
                    tile_data = np.clip(tile_data * 255, 0, 255).astype(np.uint8)
                
                # Keep only RGB
                if tile_data.shape[-1] > 3:
                    tile_data = tile_data[:, :, :3]
                
                # Process with DCEC
                cluster_map = process_tile_with_dcec(
                    tile_data,
                    feature_extractor,
                    n_clusters=n_clusters,
                    pretrain_epochs=pretrain_epochs,
                    train_epochs=train_epochs,
                    device=feature_extractor.device
                )
                
                # Save results
                tile_name = f"tile_r{tile_row}_c{tile_col}"
                
                save_geotiff_with_window(
                    output_dir / f"{tile_name}_original.tif",
                    tile_data,
                    src,
                    row_off,
                    col_off,
                    description=f"Original tile {tile_idx}"
                )
                
                save_geotiff_with_window(
                    output_dir / f"{tile_name}_segmentation.tif",
                    cluster_map,
                    src,
                    row_off,
                    col_off,
                    description=f"DCEC segmentation with {n_clusters} clusters"
                )
                
                # Visualize
                if visualize:
                    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
                    
                    axes[0].imshow(tile_data)
                    axes[0].set_title(f'Original - Tile {tile_idx}')
                    axes[0].axis('off')
                    
                    axes[1].imshow(cluster_map, cmap='tab20')
                    axes[1].set_title(f'DCEC Segmentation ({n_clusters} clusters)')
                    axes[1].axis('off')
                    
                    plt.tight_layout()
                    plt.savefig(
                        output_dir / f"{tile_name}_visualization.png",
                        dpi=150,
                        bbox_inches='tight'
                    )
                    plt.close()
                
                print(f"  ✓ Saved: {tile_name}")
    
    print("\n" + "=" * 70)
    print("✅ Processing Complete!")
    print(f"  Processed {total_tiles} tiles")
    print(f"  Output: {output_dir}")
    print("=" * 70)

if __name__ == "__main__":
    CONFIG = {
        'tiff_path': '/content/drive/MyDrive/BlueCARES_Seagrass/orthophoto/processed_october24_site_2_100m_transparent_mosaic_group1.tif',
        'output_dir': '/content/drive/MyDrive/BlueCARES_Seagrass/out/dcec_segmentation_cluster20',
        'tile_size': 4096,
        'n_clusters': 20,
        'model_size': 'large',
        'pretrain_epochs': 200,
        'train_epochs': 100,
        'visualize': True
    }
    
    process_large_orthomosaic_dcec(**CONFIG)

🚀 DINOv2 + DCEC (Original) Segmentation Pipeline
🔄 Loading DINOv2-large...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


✓ Model loaded successfully on cuda

📂 Opening: processed_october24_site_2_100m_transparent_mosaic_group1.tif


RasterioIOError: /content/drive/MyDrive/BlueCARES_Seagrass/orthophoto/processed_october24_site_2_100m_transparent_mosaic_group1.tif: No such file or directory